这里计算所有hindcast的数据并且进行保存，JFM FMA MAM AMJ等等。然后不同的hindcast有不同的seasons。保存为Z3  LAT LON 平均场，数据结构是 member，season，lat，lon

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockA — Hindcast Z500(lat,lon) per member seasonal means (case-wise NetCDF)
============================================================================

目标
----
对每个 hindcast case（如 0008-02 / 0008-02V2 / 0008-02-nocouple ...），
读取已经由 sh 提取出的 Z3(member) 文件，计算 Z500 的 3-month mean 场：

  Seasons: JFM, FMA, MAM, AMJ  (每个都是 3 个月)
  BUT: 每个 case 的时间起止可能不同，每个 member 也可能缺段
  -> 对每个 member，只有当该 season 的 3 个自然月均至少有 1 个 timestep 才认为可算
  -> 不可算则该 member-season 的 Z500 场填 NaN，并标记 season_valid=0

输入
----
/home/weiji/restart_exam/hindcast_data/<CASE>/Z3/*.Z3.nc
  - 每个文件一个 member
  - 变量名为 Z3
  - 已经 pressure-interp 到标准压层（plev in Pa）

输出
----
/home/weiji/restart_exam/code/2060202HINDCAST/result/Z500_fields/Z500_fields_<CASE>.nc

输出变量
--------
- Z500_season(member, season, lat, lon) : float32 (单位继承原始 Z3)
- season_valid(member, season)          : int8 (1=该 member-season 可算, 0=不可算)
- month_counts(member, season, mindex)  : int16 (每个 season 的 3 个月分别有多少 timestep)
- season_months(season, mindex)         : int8  (该 season 的三个月份编号，如 FMA=[2,3,4])

attrs
-----
包含 case_name / family / group_label / 规则说明等

打印输出
--------
- 每个 case 的 member 时间长度分布（min/p25/median/p75/max）
- 每个 season 的 valid_members 数量
- 最终写出的文件路径
"""

import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import xarray as xr


# ===================== 配置区 =====================
HINDCAST_BASE_DIR = Path("/home/weiji/restart_exam/hindcast_data")

OUT_DIR = Path("/home/weiji/restart_exam/code/2060202HINDCAST/result/Z500_fields")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 只处理某些年份；None 表示全部年份
TARGET_YEARS: Optional[List[str]] = ["0003", "0008", "0013", "0014", "0019"]

# 变量目录/变量名
VAR_DIRNAME = "Z3"
VAR_NAME = "Z3"

# Z500 目标压层（Pa）
PLEV_TARGET_PA = 50000.0

# seasons (3-month windows)
SEASONS: Dict[str, Tuple[int, int, int]] = {
    "JFM": (1, 2, 3),
    "FMA": (2, 3, 4),
    "MAM": (3, 4, 5),
    "AMJ": (4, 5, 6),
}
SEASON_ORDER = ["JFM", "FMA", "MAM", "AMJ"]  # 固定顺序写入文件

# NetCDF 压缩设置（建议开启，不然文件很大）
COMPRESS_LEVEL = 4
USE_FLOAT32 = True
# ==================================================


MONTH_NAME = {
    "01": "Jan", "02": "Feb", "03": "Mar", "04": "Apr",
    "05": "May", "06": "Jun", "07": "Jul", "08": "Aug",
    "09": "Sep", "10": "Oct", "11": "Nov", "12": "Dec",
}

BASE_FAMILIES: Dict[str, Dict[str, str]] = {
    "":   {"family": "small_perturbation", "short": "SP"},
    "V2": {"family": "large_perturbation", "short": "LP"},
    "V3": {"family": "q_perturbation",     "short": "QP"},
}


@dataclass
class CaseMeta:
    case_name: str
    year_str: str
    month_str: str
    suffix: str
    family: str
    family_short: str
    group_label: str


def parse_case_name(case_name: str) -> CaseMeta:
    """
    支持：0008-02, 0008-02V2, 0008-02V3, 0008-02-nocouple 等
    """
    m = re.match(r"^(?P<year>\d{4})-(?P<month>\d{2})(?P<suffix>.*)$", case_name)
    if not m:
        raise ValueError(f"Bad case name: {case_name}")

    year_str = m.group("year")
    month_str = m.group("month")
    suffix = m.group("suffix")  # "", "V2", "V3", "-nocouple" ...

    month_label = MONTH_NAME.get(month_str, month_str)

    # family classification
    family = "unknown"
    family_short = "UNK"
    if suffix in BASE_FAMILIES:
        family = BASE_FAMILIES[suffix]["family"]
        family_short = BASE_FAMILIES[suffix]["short"]
    elif "nocouple" in suffix.lower():
        family = "initial_prescribed_o3"
        family_short = "NC"

    # group_label（沿用你 BlockA 的命名风格）
    if family == "small_perturbation" and "nocouple" not in suffix.lower():
        group_label = f"{month_label}_couple"
    elif family == "initial_prescribed_o3":
        group_label = f"{month_label}_initial_prescribeO3"
    elif family == "large_perturbation":
        group_label = f"{month_label}_largePert"
    elif family == "q_perturbation":
        group_label = f"{month_label}_Qpert"
    else:
        group_label = case_name

    return CaseMeta(
        case_name=case_name,
        year_str=year_str,
        month_str=month_str,
        suffix=suffix,
        family=family,
        family_short=family_short,
        group_label=group_label,
    )


def list_case_dirs(base_dir: Path) -> List[Path]:
    out = []
    for p in sorted(base_dir.iterdir()):
        if p.is_dir() and re.match(r"^\d{4}-\d{2}.*$", p.name):
            out.append(p)
    return out


def list_member_files(case_dir: Path) -> List[Path]:
    vdir = case_dir / VAR_DIRNAME
    if not vdir.is_dir():
        return []
    # 你 sh 输出的后缀是 ".Z3.nc"
    return sorted(vdir.glob(f"*.{VAR_NAME}.nc"))


def parse_member_id_from_filename(f: Path, case_name: str, fallback_idx: int) -> int:
    """
    尽量复用你之前的逻辑：文件名按 '.' split，
    找到 token == case_name，下一段如果是纯数字，则为 member id。

    否则 fallback: fallback_idx (从1开始)
    """
    parts = f.name.split(".")
    if case_name in parts:
        i = parts.index(case_name)
        if i + 1 < len(parts) and parts[i + 1].isdigit():
            try:
                return int(parts[i + 1])
            except Exception:
                pass
    return fallback_idx


def unify_vertical_dim(da: xr.DataArray) -> xr.DataArray:
    """
    统一垂直维为 plev
    """
    dims = da.dims
    if ("plev_2" in dims) and ("plev" not in dims):
        da = da.rename({"plev_2": "plev"})
    if ("lev" in dims) and ("plev" not in dims):
        da = da.rename({"lev": "plev"})
    if ("level" in dims) and ("plev" not in dims):
        da = da.rename({"level": "plev"})
    # 极少见：同时存在 plev 和 plev_2，默认丢弃 plev_2
    if ("plev" in da.dims) and ("plev_2" in da.dims):
        da = da.isel(plev_2=0, drop=True)
    return da


def load_case_z3(case_dir: Path) -> Tuple[Optional[xr.DataArray], Optional[CaseMeta]]:
    """
    加载一个 case 的所有 member Z3，拼成：
      Z3(member, time, plev, lat, lon)

    返回 (DataArray, CaseMeta)
    """
    case_name = case_dir.name
    meta = parse_case_name(case_name)

    files = list_member_files(case_dir)
    if not files:
        return None, None

    da_list = []

    for idx, f in enumerate(files, start=1):
        ds = xr.open_dataset(f, decode_times=True)
        if VAR_NAME not in ds.data_vars:
            ds.close()
            continue

        da = unify_vertical_dim(ds[VAR_NAME])
        ds.close()

        mid = parse_member_id_from_filename(f, case_name, fallback_idx=idx)
        da = da.expand_dims("member").assign_coords(member=[mid])

        da_list.append(da)

    if not da_list:
        return None, None

    # 尽量保持坐标一致；如出现网格不一致，会走 outer 并出现 NaN
    da_all = xr.concat(da_list, dim="member", join="outer")

    # 排序 member
    da_all = da_all.sortby("member")
    return da_all, meta


def finite_time_mask(z_m: xr.DataArray) -> xr.DataArray:
    """
    给定 z_m(time, lat, lon)，返回每个 time 是否有至少一个有限值（避免全 NaN 的时间点被计入）
    """
    dims_to_any = [d for d in z_m.dims if d != "time"]
    return xr.apply_ufunc(np.isfinite, z_m).any(dim=dims_to_any)


def member_season_mean_field(
    z_m: xr.DataArray, months: Tuple[int, int, int]
) -> Tuple[bool, xr.DataArray, Tuple[int, int, int]]:
    """
    对单个 member 的 z_m(time,lat,lon)，计算某个 season 的均值场。
    要求：该 season 的 3 个月份各至少有 1 个有效 timestep（非全NaN）

    返回：
      ok(bool),
      mean_field(lat,lon) （ok=False 时返回全 NaN 场）
      month_counts (c1,c2,c3) 对应三个月份各自 timestep 数
    """
    valid_t = finite_time_mask(z_m)
    mon = z_m["time"].dt.month

    m1, m2, m3 = months
    c1 = int(((mon == m1) & valid_t).sum().values)
    c2 = int(((mon == m2) & valid_t).sum().values)
    c3 = int(((mon == m3) & valid_t).sum().values)

    ok = (c1 > 0) and (c2 > 0) and (c3 > 0)
    if not ok:
        nan_field = xr.full_like(z_m.isel(time=0, drop=True), np.nan)
        return False, nan_field, (c1, c2, c3)

    mask_all = ((mon == m1) | (mon == m2) | (mon == m3)) & valid_t
    mean_field = z_m.where(mask_all, drop=True).mean("time")
    return True, mean_field, (c1, c2, c3)


def describe_lengths(lengths: np.ndarray) -> str:
    """
    输出 nsteps 的分布描述
    """
    if lengths.size == 0:
        return "empty"
    p25 = int(np.percentile(lengths, 25))
    p75 = int(np.percentile(lengths, 75))
    return f"min={lengths.min()} p25={p25} median={int(np.median(lengths))} p75={p75} max={lengths.max()}"


def main():
    case_dirs = list_case_dirs(HINDCAST_BASE_DIR)

    if TARGET_YEARS is not None:
        target_set = set(TARGET_YEARS)
        # case name like "0008-02V2" -> year token is before first "-"
        case_dirs = [p for p in case_dirs if p.name.split("-")[0] in target_set]

    print(f"[INFO] Found {len(case_dirs)} cases under {HINDCAST_BASE_DIR} (years={TARGET_YEARS})")
    print(f"[INFO] Output dir: {OUT_DIR}")

    for case_dir in case_dirs:
        case_name = case_dir.name
        out_path = OUT_DIR / f"Z500_fields_{case_name}.nc"

        if out_path.exists():
            print(f"[SKIP] Exists: {out_path}")
            continue

        da_z3, meta = load_case_z3(case_dir)
        if da_z3 is None or meta is None:
            print(f"[SKIP] {case_name}: no valid Z3 member files.")
            continue

        if "plev" not in da_z3.dims:
            print(f"[WARN] {case_name}: no 'plev' dimension found. Skip.")
            continue

        # 选 Z500（nearest）
        z500 = da_z3.sel(plev=PLEV_TARGET_PA, method="nearest")  # (member,time,lat,lon)

        # 计算每个 member 的有效时间步长（排除全 NaN time）
        member_vals = z500["member"].values
        lengths = []
        for mid in member_vals:
            z_m = z500.sel(member=mid)
            valid_t = finite_time_mask(z_m)
            lengths.append(int(valid_t.sum().values))
        lengths_arr = np.asarray(lengths, dtype=int)

        print("\n" + "=" * 70)
        print(f"[CASE] {case_name}  group={meta.group_label}  family={meta.family_short}")
        print(f"  members={len(member_vals)}")
        print(f"  time-length distribution (n_valid_timesteps): {describe_lengths(lengths_arr)}")

        # 预分配输出数组
        seasons = np.array(SEASON_ORDER, dtype="U3")
        nmem = len(member_vals)
        nsea = len(seasons)

        # lat/lon coords
        lat = z500["lat"]
        lon = z500["lon"] if "lon" in z500.coords else None

        # Z500_season: (member, season, lat, lon)
        if lon is not None:
            z500_season = xr.DataArray(
                np.full((nmem, nsea, lat.size, lon.size), np.nan,
                        dtype=np.float32 if USE_FLOAT32 else np.float64),
                dims=("member", "season", "lat", "lon"),
                coords={"member": member_vals, "season": seasons, "lat": lat, "lon": lon},
                name="Z500_season",
            )
        else:
            z500_season = xr.DataArray(
                np.full((nmem, nsea, lat.size), np.nan,
                        dtype=np.float32 if USE_FLOAT32 else np.float64),
                dims=("member", "season", "lat"),
                coords={"member": member_vals, "season": seasons, "lat": lat},
                name="Z500_season",
            )

        season_valid = xr.DataArray(
            np.zeros((nmem, nsea), dtype=np.int8),
            dims=("member", "season"),
            coords={"member": member_vals, "season": seasons},
            name="season_valid",
        )

        month_counts = xr.DataArray(
            np.zeros((nmem, nsea, 3), dtype=np.int16),
            dims=("member", "season", "mindex"),
            coords={"member": member_vals, "season": seasons, "mindex": [0, 1, 2]},
            name="month_counts",
        )

        season_months = xr.DataArray(
            np.array([SEASONS[s] for s in SEASON_ORDER], dtype=np.int8),
            dims=("season", "mindex"),
            coords={"season": seasons, "mindex": [0, 1, 2]},
            name="season_months",
        )

        # 逐 member 逐 season 计算
        valid_members_count = {s: 0 for s in SEASON_ORDER}

        for im, mid in enumerate(member_vals):
            z_m = z500.sel(member=mid)  # (time,lat,lon)
            for isea, s in enumerate(SEASON_ORDER):
                months = SEASONS[s]
                ok, mean_field, (c1, c2, c3) = member_season_mean_field(z_m, months)

                month_counts[im, isea, 0] = c1
                month_counts[im, isea, 1] = c2
                month_counts[im, isea, 2] = c3

                if ok:
                    season_valid[im, isea] = 1
                    valid_members_count[s] += 1

                    if lon is not None:
                        z500_season[im, isea, :, :] = mean_field.astype(z500_season.dtype)
                    else:
                        z500_season[im, isea, :] = mean_field.astype(z500_season.dtype)

        # 打印每个 season 的有效成员数
        for s in SEASON_ORDER:
            print(f"  season {s}: valid_members={valid_members_count[s]}/{nmem}")

        # 组装 Dataset
        ds_out = xr.Dataset(
            data_vars={
                "Z500_season": z500_season,
                "season_valid": season_valid,
                "month_counts": month_counts,
                "season_months": season_months,
            },
            attrs={
                "case_name": meta.case_name,
                "year_str": meta.year_str,
                "month_str": meta.month_str,
                "suffix": meta.suffix,
                "family": meta.family,
                "family_short": meta.family_short,
                "group_label": meta.group_label,
                "plev_target_pa": float(PLEV_TARGET_PA),
                "season_rule": (
                    "A member-season is computed only if each of the three calendar months "
                    "has >=1 valid timestep (not all-NaN over lat/lon). Otherwise NaN."
                ),
                "note": (
                    "Per-case output. Contains per-member seasonal mean Z500 fields "
                    "for JFM/FMA/MAM/AMJ. Missing seasons for a member are NaN with season_valid=0."
                ),
            },
        )

        # 单位继承（如果原始有 units）
        if "units" in z500.attrs:
            ds_out["Z500_season"].attrs["units"] = z500.attrs["units"]

        # 写 NetCDF（压缩）
        encoding = {
            "Z500_season": {"zlib": True, "complevel": COMPRESS_LEVEL},
            "season_valid": {"zlib": True, "complevel": COMPRESS_LEVEL},
            "month_counts": {"zlib": True, "complevel": COMPRESS_LEVEL},
            "season_months": {"zlib": True, "complevel": COMPRESS_LEVEL},
        }

        ds_out.to_netcdf(out_path, encoding=encoding)
        print(f"[SAVE] -> {out_path}")

    print("\n[INFO] All done.")


if __name__ == "__main__":
    main()


这里处理气候态BW2000CN和longrun的数据

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, glob
import numpy as np
import xarray as xr

# ============================================================
#  Build reference Z500 seasonal means:
#   (A) Climatology from B2000WCN-withchem extracted hybrid Z3 yearly files
#       -> monthly mean -> stdplev -> 12-month climatology -> seasonal means
#   (B) Longrun BWCN.002 h3 history files
#       -> monthly mean -> stdplev -> seasonal means per-year
#
#  Seasons: JFM, FMA, MAM, AMJ
#  Outputs to: /home/weiji/restart_exam/code/2060202HINDCAST/result/Z500_fields
# ============================================================

# ======================== paths =========================
OUT_DIR = "/home/weiji/restart_exam/code/2060202HINDCAST/result/Z500_fields"
os.makedirs(OUT_DIR, exist_ok=True)

# (A) climatology source (same as your example program)
B2000_Z3_HYB_DIR = "/home/weiji/restart_exam/longrun_B2000WCN_withchem_data/Z3"
OUT_CLIM_SEASON  = os.path.join(OUT_DIR, "Z500_climatology_season_12mon.nc")

# (B) longrun source (same as your example program)
BWCN_H3_DIR      = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"
OUT_LONGRUN_SEAS = os.path.join(OUT_DIR, "Z500_longrun_season_yearly.nc")

# optional: if you only want some model-years (as you mentioned 0008/0003/0013/0014/0019)
# set to None to keep all years
KEEP_MODEL_YEARS = [3, 8, 13, 14, 19]  # or None

# ===================== standard plev =====================
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

PLEV_TARGET_PA = 50000.0  # Z500

# ===================== seasons ===========================
SEASONS = {
    "JFM": (1, 2, 3),
    "FMA": (2, 3, 4),
    "MAM": (3, 4, 5),
    "AMJ": (4, 5, 6),
}
SEASON_ORDER = ["JFM", "FMA", "MAM", "AMJ"]

# ===================== IO strategy =======================
ENGINE   = "netcdf4"
PARALLEL = False
CHUNKS   = {"time": 1}

COMPRESS_LEVEL = 4
USE_FLOAT32 = True

# =========================================================
def safe_open_mfdataset(files):
    return xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=PARALLEL,
        use_cftime=True,
        chunks=CHUNKS,
        engine=ENGINE,
    )

def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    """p_mid = hyam*P0 + hybm*PS  (Pa)"""
    hyam = ds["hyam"]  # (lev)
    hybm = ds["hybm"]  # (lev)
    P0   = ds["P0"]    # scalar Pa
    PS   = ds["PS"]    # (time, lat, lon)
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    p_mid.attrs["units"] = "Pa"
    return p_mid

def interp_profile_logp(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt: np.ndarray) -> xr.DataArray:
    """
    log(p) linear interpolation for each vertical column to target pressure levels.
    v_hyb, p_hyb: (..., lev), p in Pa
    returns: (..., plev)
    """
    p_tgt = np.asarray(p_tgt, float)
    nplev = int(p_tgt.size)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full((nplev,), np.nan, float)
        p_use = pcol[m]
        v_use = vcol[m]
        idx = np.argsort(p_use)  # ascending pressure
        p_use = p_use[idx]
        v_use = v_use[idx]
        return np.interp(np.log(p_tgt), np.log(p_use), v_use, left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={"plev": nplev},
    )
    out = out.assign_coords(plev=("plev", p_tgt))
    out["plev"].attrs["units"] = "Pa"
    return out

def monthly_mean_then_interp_to_stdplev(ds: xr.Dataset, varname="Z3") -> xr.DataArray:
    need = [varname, "PS", "hyam", "hybm", "P0"]
    missing = [v for v in need if v not in ds.variables]
    if missing:
        raise RuntimeError(f"Dataset missing vars: {missing}")

    ds_m = ds[need].resample(time="MS").mean()
    v_m = ds_m[varname]               # (time, lev, lat, lon)
    p_m = compute_pressure_mid(ds_m)  # (time, lev, lat, lon)

    v_std = interp_profile_logp(v_m, p_m, PLEV_STD_PA)
    v_std = v_std.transpose("time", "plev", "lat", "lon")

    v_std.name = f"{varname}_stdplev"
    v_std.attrs.update(
        units=v_m.attrs.get("units", ""),
        long_name=f"{varname}: monthly mean then log(p) interpolated to standard pressure levels",
    )
    return v_std

def seasonal_mean_from_monthly_field(v_monthly: xr.DataArray, months: tuple) -> xr.DataArray:
    """
    v_monthly: (time, lat, lon) monthly means (already Z500)
    Return: yearly seasonal mean (year, lat, lon) with completeness check:
      each year must contain all 3 months to be valid, else NaN.
    """
    m1, m2, m3 = months
    mon = v_monthly["time"].dt.month
    yr  = v_monthly["time"].dt.year

    out_list = []
    years = np.unique(yr.values)

    for y in years:
        sel = v_monthly.where(yr == y, drop=True)
        sel_mon = sel["time"].dt.month
        c1 = int((sel_mon == m1).sum().values)
        c2 = int((sel_mon == m2).sum().values)
        c3 = int((sel_mon == m3).sum().values)

        if (c1 > 0) and (c2 > 0) and (c3 > 0):
            msel = sel.where((sel_mon == m1) | (sel_mon == m2) | (sel_mon == m3), drop=True)
            mean_y = msel.mean("time")
        else:
            mean_y = xr.full_like(sel.isel(time=0, drop=True), np.nan)

        mean_y = mean_y.expand_dims("year").assign_coords(year=[int(y)])
        out_list.append(mean_y)

    out = xr.concat(out_list, dim="year")
    return out

def write_dataset(ds: xr.Dataset, out_path: str, compress_level=4):
    enc = {}
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating):
            enc[v] = {"zlib": True, "complevel": int(compress_level),
                      "dtype": "float32" if USE_FLOAT32 else str(ds[v].dtype),
                      "_FillValue": np.float32(np.nan)}
        else:
            enc[v] = {"zlib": True, "complevel": int(compress_level)}
    # coords dtype keep
    for c in ds.coords:
        if c in enc:  # unlikely
            continue
        if ds[c].dtype.kind in ["f"]:
            enc[c] = {"dtype": "float64"}
        elif ds[c].dtype.kind in ["i", "u"]:
            enc[c] = {"dtype": "int32"}
    ds.to_netcdf(out_path, encoding=enc)

# =========================================================
# (A) Climatology: 12-month climatology -> seasonal means
# =========================================================
print("[A] Building climatology seasonal means from:", B2000_Z3_HYB_DIR)

b2000_files = sorted(glob.glob(os.path.join(B2000_Z3_HYB_DIR, "*.Z3.hybrid.nc")))
if len(b2000_files) == 0:
    raise RuntimeError(f"No B2000 hybrid Z3 files found: {B2000_Z3_HYB_DIR}")

ds_b2000 = safe_open_mfdataset(b2000_files)
z3_b2000_monthly_std = monthly_mean_then_interp_to_stdplev(ds_b2000, varname="Z3")  # (time,plev,lat,lon)
ds_b2000.close()

z500_b2000_monthly = z3_b2000_monthly_std.sel(plev=PLEV_TARGET_PA, method="nearest")  # (time,lat,lon)

# 12-month climatology (month_of_year=1..12)
z500_clim12 = z500_b2000_monthly.groupby("time.month").mean("time").rename({"month": "month_of_year"})
z500_clim12["month_of_year"].attrs["long_name"] = "Month of year (1-12)"

# seasonal mean using climatology months
season_labels = np.array(SEASON_ORDER, dtype="U3")
clim_season = []
for s in SEASON_ORDER:
    m1, m2, m3 = SEASONS[s]
    mean_s = z500_clim12.sel(month_of_year=[m1, m2, m3]).mean("month_of_year")
    mean_s = mean_s.expand_dims("season").assign_coords(season=[s])
    clim_season.append(mean_s)
Z500_clim_season = xr.concat(clim_season, dim="season")  # (season,lat,lon)

ds_clim_out = xr.Dataset(
    data_vars={"Z500_clim_season": Z500_clim_season.astype(np.float32 if USE_FLOAT32 else np.float64)},
    coords={"season": season_labels, "lat": Z500_clim_season["lat"], "lon": Z500_clim_season["lon"]},
    attrs={
        "source": "B2000WCN_withchem longrun extracted hybrid Z3",
        "processing": "monthly mean -> stdplev log(p) interp -> Z500 -> 12-month climatology -> 3-month seasonal means",
        "seasons": "JFM,FMA,MAM,AMJ",
        "plev_target_pa": float(PLEV_TARGET_PA),
    },
)
if "units" in z3_b2000_monthly_std.attrs:
    ds_clim_out["Z500_clim_season"].attrs["units"] = z3_b2000_monthly_std.attrs["units"]

write_dataset(ds_clim_out, OUT_CLIM_SEASON, compress_level=COMPRESS_LEVEL)
print("[SAVE] climatology seasons ->", OUT_CLIM_SEASON)

# =========================================================
# (B) Longrun: yearly seasonal means from BWCN h3
# =========================================================
print("\n[B] Building longrun seasonal means from:", BWCN_H3_DIR)

bwcn_files = sorted(glob.glob(os.path.join(BWCN_H3_DIR, "*.cam.h3.*.nc")))
if len(bwcn_files) == 0:
    raise RuntimeError(f"No BWCN h3 files found: {BWCN_H3_DIR}")

ds_bwcn = safe_open_mfdataset(bwcn_files)
z3_bwcn_monthly_std = monthly_mean_then_interp_to_stdplev(ds_bwcn, varname="Z3")  # (time,plev,lat,lon)
ds_bwcn.close()

z500_bwcn_monthly = z3_bwcn_monthly_std.sel(plev=PLEV_TARGET_PA, method="nearest")  # (time,lat,lon)

# optionally filter years
if KEEP_MODEL_YEARS is not None:
    years_all = z500_bwcn_monthly["time"].dt.year
    z500_bwcn_monthly = z500_bwcn_monthly.where(years_all.isin(KEEP_MODEL_YEARS), drop=True)

# compute yearly seasonal means for each season
season_fields = []
for s in SEASON_ORDER:
    seas_y = seasonal_mean_from_monthly_field(z500_bwcn_monthly, SEASONS[s])  # (year,lat,lon)
    seas_y = seas_y.expand_dims("season").assign_coords(season=[s])
    season_fields.append(seas_y)

Z500_longrun_season = xr.concat(season_fields, dim="season")  # (season,year,lat,lon)
Z500_longrun_season = Z500_longrun_season.transpose("year", "season", "lat", "lon")

ds_longrun_out = xr.Dataset(
    data_vars={"Z500_longrun_season": Z500_longrun_season.astype(np.float32 if USE_FLOAT32 else np.float64)},
    coords={
        "year": Z500_longrun_season["year"].astype(np.int32),
        "season": season_labels,
        "lat": Z500_longrun_season["lat"],
        "lon": Z500_longrun_season["lon"],
    },
    attrs={
        "source": "BWCN.e122.f19_g16.002 atm/hist cam.h3",
        "processing": "monthly mean -> stdplev log(p) interp -> Z500 -> yearly 3-month seasonal means with completeness check",
        "seasons": "JFM,FMA,MAM,AMJ",
        "plev_target_pa": float(PLEV_TARGET_PA),
        "year_filter": str(KEEP_MODEL_YEARS),
        "season_rule": "For a given (year, season), must contain all 3 calendar months; otherwise NaN.",
    },
)
if "units" in z3_bwcn_monthly_std.attrs:
    ds_longrun_out["Z500_longrun_season"].attrs["units"] = z3_bwcn_monthly_std.attrs["units"]

write_dataset(ds_longrun_out, OUT_LONGRUN_SEAS, compress_level=COMPRESS_LEVEL)
print("[SAVE] longrun yearly seasons ->", OUT_LONGRUN_SEAS)

print("\n[INFO] Done.")


根据气候态和hindcast/longrun数据计算anomaly和对应的ACC。

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import re
from pathlib import Path
import numpy as np
import xarray as xr
import pandas as pd

# ============================================================
#  BlockB — Compute anomalies + ACC (hindcast vs longrun ref)
#
#  Inputs (produced by your previous blocks):
#    - Hindcast per-case:
#        Z500_fields_<CASE>.nc
#        vars: Z500_season(member,season,lat,lon), season_valid(member,season)
#    - Climatology:
#        Z500_climatology_season_12mon.nc
#        var: Z500_clim_season(season,lat,lon)
#    - Longrun (reference):
#        Z500_longrun_season_yearly.nc
#        var: Z500_longrun_season(year,season,lat,lon)
#
#  Outputs:
#    - Hindcast anomalies (per case):
#        Z500_anom_fields_<CASE>.nc
#    - Longrun anomalies:
#        Z500_longrun_anom_yearly.nc
#    - ACC per case:
#        ACC_<CASE>.nc  + a combined ACC_summary.csv
#
#  ACC definition:
#    Weighted pattern correlation over lat/lon using cos(lat) weights.
# ============================================================

# ===================== paths =====================
BASE_DIR = Path("/home/weiji/restart_exam/code/2060202HINDCAST/result/Z500_fields")

HIND_DIR = BASE_DIR                     # contains Z500_fields_<CASE>.nc
CLIM_FILE = BASE_DIR / "Z500_climatology_season_12mon.nc"
LONGRUN_FILE = BASE_DIR / "Z500_longrun_season_yearly.nc"

OUT_ANOM_HIND_DIR = BASE_DIR / "Z500_anom_fields"
OUT_ANOM_REF_DIR  = BASE_DIR / "Z500_anom_ref"
OUT_ACC_DIR       = BASE_DIR / "ACC"

OUT_ANOM_HIND_DIR.mkdir(parents=True, exist_ok=True)
OUT_ANOM_REF_DIR.mkdir(parents=True, exist_ok=True)
OUT_ACC_DIR.mkdir(parents=True, exist_ok=True)

OUT_REF_ANOM_FILE = OUT_ANOM_REF_DIR / "Z500_longrun_anom_yearly.nc"
OUT_ACC_CSV = OUT_ACC_DIR / "ACC_summary.csv"

# Which seasons to process (must match your files)
SEASON_ORDER = ["JFM", "FMA", "MAM", "AMJ"]

# If you only want certain hindcast model-years
TARGET_YEARS = {"0003", "0008", "0013", "0014", "0019"}  # set to None for all
# TARGET_YEARS = None

COMPRESS_LEVEL = 4
USE_FLOAT32 = True

# ===================== helpers =====================
def _to_float32_if(x):
    return x.astype(np.float32) if USE_FLOAT32 else x

def parse_year_from_case(case_name: str) -> int:
    """
    case_name like '0008-02', '0013-11V2', '0003-02-nocouple' -> year int 8,13,3
    """
    m = re.match(r"^(?P<year>\d{4})-", case_name)
    if not m:
        raise ValueError(f"Cannot parse year from case_name={case_name}")
    return int(m.group("year"))

def coslat_weights(lat: xr.DataArray) -> xr.DataArray:
    w = np.cos(np.deg2rad(lat))
    # clamp tiny negatives due to precision
    w = xr.DataArray(np.maximum(w, 0.0), coords={"lat": lat}, dims=("lat",))
    return w

def weighted_corr2d(a2d: np.ndarray, b2d: np.ndarray, w_lat: np.ndarray) -> float:
    """
    Weighted correlation over lat/lon.
    a2d, b2d: (lat, lon) arrays with NaNs allowed.
    w_lat: (lat,) weights = cos(lat)

    Returns NaN if insufficient valid points.
    """
    if a2d.ndim != 2 or b2d.ndim != 2:
        raise ValueError("a2d/b2d must be 2D (lat,lon).")

    lat_n, lon_n = a2d.shape
    if b2d.shape != (lat_n, lon_n):
        raise ValueError("a2d and b2d must have same shape.")

    # expand weights to (lat,lon)
    w2d = np.repeat(w_lat.reshape(lat_n, 1), lon_n, axis=1)

    m = np.isfinite(a2d) & np.isfinite(b2d) & np.isfinite(w2d) & (w2d > 0)
    if m.sum() < 10:
        return np.nan

    aa = a2d[m]
    bb = b2d[m]
    ww = w2d[m]

    # weighted mean
    wsum = ww.sum()
    if wsum <= 0:
        return np.nan
    am = (ww * aa).sum() / wsum
    bm = (ww * bb).sum() / wsum

    a0 = aa - am
    b0 = bb - bm

    cov = (ww * a0 * b0).sum()
    va  = (ww * a0 * a0).sum()
    vb  = (ww * b0 * b0).sum()

    if va <= 0 or vb <= 0:
        return np.nan

    return float(cov / np.sqrt(va * vb))

def list_hindcast_case_files(hind_dir: Path):
    files = sorted(hind_dir.glob("Z500_fields_*.nc"))
    out = []
    for f in files:
        case = f.name.replace("Z500_fields_", "").replace(".nc", "")
        if TARGET_YEARS is not None:
            y4 = case.split("-")[0]
            if y4 not in TARGET_YEARS:
                continue
        out.append((case, f))
    return out

def write_netcdf(ds: xr.Dataset, out_path: Path):
    enc = {}
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating):
            enc[v] = {
                "zlib": True, "complevel": int(COMPRESS_LEVEL),
                "dtype": "float32" if USE_FLOAT32 else str(ds[v].dtype),
                "_FillValue": np.float32(np.nan),
            }
        else:
            enc[v] = {"zlib": True, "complevel": int(COMPRESS_LEVEL)}
    ds.to_netcdf(out_path, encoding=enc)

# ===================== main =====================
def main():
    # ---- load climatology ----
    if not CLIM_FILE.exists():
        raise FileNotFoundError(f"Missing climatology file: {CLIM_FILE}")
    ds_clim = xr.open_dataset(CLIM_FILE)
    if "Z500_clim_season" not in ds_clim:
        raise KeyError("Climatology file must contain variable 'Z500_clim_season'")
    clim = ds_clim["Z500_clim_season"].sel(season=SEASON_ORDER)  # (season,lat,lon)
    # ensure coords exist
    lat = clim["lat"]
    lon = clim["lon"]
    w_lat = coslat_weights(lat).values
    ds_clim.close()

    # ---- load longrun ----
    if not LONGRUN_FILE.exists():
        raise FileNotFoundError(f"Missing longrun file: {LONGRUN_FILE}")
    ds_lr = xr.open_dataset(LONGRUN_FILE)
    if "Z500_longrun_season" not in ds_lr:
        raise KeyError("Longrun file must contain variable 'Z500_longrun_season'")
    lr = ds_lr["Z500_longrun_season"].sel(season=SEASON_ORDER)  # (year,season,lat,lon)

    # ---- compute longrun anomalies once and save ----
    # broadcast clim(season,lat,lon) to (year,season,lat,lon)
    lr_anom = lr - clim
    lr_anom.name = "Z500_longrun_anom"
    ds_lr_anom = xr.Dataset(
        data_vars={"Z500_longrun_anom": _to_float32_if(lr_anom)},
        coords={
            "year": lr_anom["year"].astype(np.int32),
            "season": lr_anom["season"].astype("U3"),
            "lat": lr_anom["lat"],
            "lon": lr_anom["lon"],
        },
        attrs={
            "note": "Longrun (BWCN) Z500 seasonal anomalies relative to B2000WCN-withchem 12-month climatology seasonal means.",
            "climatology_file": str(CLIM_FILE),
            "longrun_file": str(LONGRUN_FILE),
            "seasons": ",".join(SEASON_ORDER),
        },
    )
    if OUT_REF_ANOM_FILE.exists():
        print(f"[SKIP] Ref anomaly exists: {OUT_REF_ANOM_FILE}")
    else:
        write_netcdf(ds_lr_anom, OUT_REF_ANOM_FILE)
        print(f"[SAVE] Ref anomaly -> {OUT_REF_ANOM_FILE}")

    # keep ds_lr open for quick selection
    # alternatively we can just use lr_anom array already in memory
    lr_anom_inmem = lr_anom

    # ---- process each hindcast case ----
    case_files = list_hindcast_case_files(HIND_DIR)
    print(f"[INFO] Found {len(case_files)} hindcast case files in {HIND_DIR}")

    summary_rows = []

    for case_name, fcase in case_files:
        print("\n" + "=" * 80)
        print(f"[CASE] {case_name}")
        ds_h = xr.open_dataset(fcase)

        if "Z500_season" not in ds_h:
            print(f"[SKIP] {fcase} has no Z500_season")
            ds_h.close()
            continue

        z = ds_h["Z500_season"].sel(season=SEASON_ORDER)  # (member,season,lat,lon)
        # basic coord consistency check
        if ("lat" not in z.coords) or ("lon" not in z.coords):
            print("[SKIP] Missing lat/lon coords")
            ds_h.close()
            continue

        # align to climatology grid if needed
        # (assumes same lat/lon values; if not, this will introduce NaNs)
        z = z.reindex(lat=lat, lon=lon)

        # season_valid is optional but strongly recommended
        if "season_valid" in ds_h:
            sv = ds_h["season_valid"].sel(season=SEASON_ORDER)  # (member,season)
        else:
            # if absent, treat all as valid
            sv = xr.DataArray(
                np.ones((z.sizes["member"], z.sizes["season"]), dtype=np.int8),
                dims=("member", "season"),
                coords={"member": z["member"], "season": z["season"]},
            )

        # ---- hindcast anomaly ----
        hind_anom = z - clim  # broadcast clim(season,lat,lon)
        hind_anom.name = "Z500_hindcast_anom"

        # save hindcast anomaly fields (per case)
        out_anom_case = OUT_ANOM_HIND_DIR / f"Z500_anom_fields_{case_name}.nc"
        if out_anom_case.exists():
            print(f"[SKIP] Hind anomaly exists: {out_anom_case}")
        else:
            ds_h_anom = xr.Dataset(
                data_vars={
                    "Z500_hindcast_anom": _to_float32_if(hind_anom),
                    "season_valid": sv.astype(np.int8),
                },
                coords={
                    "member": hind_anom["member"],
                    "season": hind_anom["season"].astype("U3"),
                    "lat": hind_anom["lat"],
                    "lon": hind_anom["lon"],
                },
                attrs={
                    "case_name": case_name,
                    "source_hindcast_file": str(fcase),
                    "climatology_file": str(CLIM_FILE),
                    "plev_target_pa": float(ds_h.attrs.get("plev_target_pa", 50000.0)),
                    "note": "Hindcast Z500 seasonal anomalies relative to B2000WCN-withchem 12-month climatology seasonal means.",
                },
            )
            write_netcdf(ds_h_anom, out_anom_case)
            print(f"[SAVE] Hind anomaly -> {out_anom_case}")

        # ---- ACC vs reference (year-aligned) ----
        y_int = parse_year_from_case(case_name)  # 0003 -> 3, etc.

        # reference anomaly for this hindcast-year
        # lr_anom_inmem dims: (year,season,lat,lon)
        if "year" not in lr_anom_inmem.dims:
            print("[SKIP] longrun anomaly missing year dimension")
            ds_h.close()
            continue

        if y_int not in lr_anom_inmem["year"].values:
            print(f"[WARN] reference longrun does not contain year={y_int}. ACC will be NaN for this case.")
            ref_anom_y = None
        else:
            ref_anom_y = lr_anom_inmem.sel(year=y_int)  # (season,lat,lon)

        members = hind_anom["member"].values
        seasons = hind_anom["season"].values

        acc_member = np.full((members.size, seasons.size), np.nan, dtype=np.float32 if USE_FLOAT32 else np.float64)
        acc_ens    = np.full((seasons.size,), np.nan, dtype=np.float32 if USE_FLOAT32 else np.float64)

        for isea, s in enumerate(seasons):
            # get ref field
            if ref_anom_y is None:
                continue
            ref2d = ref_anom_y.sel(season=s).values  # (lat,lon)

            # if ref is all NaN, skip
            if not np.isfinite(ref2d).any():
                continue

            # member-wise ACC
            valid_mask = sv.sel(season=s).values.astype(bool)  # (member,)
            for im, mid in enumerate(members):
                if not valid_mask[im]:
                    continue
                a2d = hind_anom.sel(member=mid, season=s).values
                acc_member[im, isea] = weighted_corr2d(a2d, ref2d, w_lat)

            # ensemble-mean ACC (mean across valid members)
            if valid_mask.any():
                ens2d = hind_anom.sel(season=s).where(sv.sel(season=s) == 1).mean("member").values
                acc_ens[isea] = weighted_corr2d(ens2d, ref2d, w_lat)

        # save ACC per case
        out_acc_case = OUT_ACC_DIR / f"ACC_{case_name}.nc"
        ds_acc = xr.Dataset(
            data_vars={
                "ACC_member": (("member", "season"), acc_member),
                "ACC_ensmean": (("season",), acc_ens),
            },
            coords={
                "member": members,
                "season": seasons.astype("U3"),
            },
            attrs={
                "case_name": case_name,
                "hindcast_year_int": int(y_int),
                "reference_year_int": int(y_int),
                "acc_definition": "Area-weighted pattern correlation of anomalies over lat/lon using cos(lat) weights.",
                "note": "ACC computed between hindcast anomaly and longrun(reference) anomaly for the matched model-year.",
                "hindcast_file": str(fcase),
                "climatology_file": str(CLIM_FILE),
                "reference_anom_file": str(OUT_REF_ANOM_FILE),
            },
        )
        write_netcdf(ds_acc, out_acc_case)
        print(f"[SAVE] ACC -> {out_acc_case}")

        # add summary rows
        # store ensemble mean ACC (one row per season)
        for isea, s in enumerate(seasons):
            summary_rows.append({
                "case": case_name,
                "year": y_int,
                "season": str(s),
                "ACC_ensmean": float(acc_ens[isea]) if np.isfinite(acc_ens[isea]) else np.nan,
                "n_valid_members": int(sv.sel(season=s).sum().values),
                "group_label": ds_h.attrs.get("group_label", ""),
                "family_short": ds_h.attrs.get("family_short", ""),
                "suffix": ds_h.attrs.get("suffix", ""),
            })

        ds_h.close()

    # ---- write summary CSV ----
    if summary_rows:
        df = pd.DataFrame(summary_rows)
        df = df.sort_values(["year", "case", "season"])
        df.to_csv(OUT_ACC_CSV, index=False)
        print("\n" + "=" * 80)
        print(f"[SAVE] ACC summary CSV -> {OUT_ACC_CSV}")
        print(df.head(10).to_string(index=False))
    else:
        print("[WARN] No cases processed; summary is empty.")

    ds_lr.close()
    print("\n[INFO] Done.")


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations

import re
from pathlib import Path
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.path as mpath


# ===================== PATHS =====================
BASE_DIR = Path("/home/weiji/restart_exam/code/2060202HINDCAST/result/Z500_fields")

HIND_ANOM_DIR = BASE_DIR / "Z500_anom_fields"
REF_ANOM_FILE = BASE_DIR / "Z500_anom_ref" / "Z500_longrun_anom_yearly.nc"
ACC_CSV       = BASE_DIR / "ACC" / "ACC_summary.csv"

HIND_FIELDS_DIR = BASE_DIR  # has Z500_fields_<CASE>.nc for attrs

OUT_FIG_DIR = BASE_DIR / "fig_lee_style_reversexy"
OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)

# ===================== CONFIG =====================
TARGET_YEARS = ["0003", "0008", "0013", "0014", "0019"]
SEASONS      = ["JFM", "FMA", "MAM", "AMJ"]

MIN_EXPS_PER_FIG = 2
MAX_MAPS_PER_ROW = 3

# --- Projection: keep previous (Polar) ---
MAP_PROJ = ccrs.NorthPolarStereo(central_longitude=0)
DATA_CRS = ccrs.PlateCarree()
POLAR_EXTENT = [0, 360, 20, 90]  # lon_min, lon_max, lat_min, lat_max

# Style
CMAP = "RdBu_r"
COASTLINE_LW = 0.5
BORDER_LW = 0.3

FIGSIZE = (12.5, 8.5)
FS_TITLE = 10
FS_LABEL = 9
FS_TICK  = 8

VLIM_Q = 0.98

# ---- BAR spacing controls (keep bar thickness + fonts unchanged) ----
BAR_HEIGHT = 0.60        # keep unchanged (thickness)
BAR_Y_SPACING = 1.25     # >1 increases spacing between bars
BAR_ROW_BASE = 0.55      # baseline height ratio for bar row
BAR_ROW_PER_BAR = 0.035  # add extra height per bar (keeps overall figsize but reallocates figure space)


# =================================================
def normalize_season(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().upper()
    if s == "APJ":
        return "AMJ"
    return s


def safe_sel_season(da: xr.DataArray, season: str) -> xr.DataArray:
    season = normalize_season(season)
    vals = [normalize_season(x) for x in da["season"].values.tolist()]
    idx = np.where(np.array(vals) == season)[0]
    if idx.size == 0:
        raise KeyError(f"Season {season} not found. Available={vals}")
    return da.isel(season=int(idx[0]))


def parse_year_int_from_case(case: str) -> int:
    m = re.match(r"^(?P<y>\d{4})-", case)
    if not m:
        raise ValueError(f"Cannot parse year from case={case}")
    return int(m.group("y"))


def list_cases_for_year(year_str: str) -> list[str]:
    patt = f"Z500_anom_fields_{year_str}-*.nc"
    out = []
    for f in sorted(HIND_ANOM_DIR.glob(patt)):
        out.append(f.name.replace("Z500_anom_fields_", "").replace(".nc", ""))
    return out


def load_acc_table() -> pd.DataFrame:
    if not ACC_CSV.exists():
        raise FileNotFoundError(f"Missing ACC summary CSV: {ACC_CSV}")
    df = pd.read_csv(ACC_CSV)
    df["season"] = df["season"].astype(str).str.upper().map(normalize_season)
    df["case"] = df["case"].astype(str)
    df["year_str"] = df["year"].apply(lambda x: f"{int(x):04d}")
    return df


def load_case_meta(case: str) -> dict:
    meta = {"case": case, "group_label": case, "family_short": "", "family": ""}
    f = HIND_FIELDS_DIR / f"Z500_fields_{case}.nc"
    if not f.exists():
        return meta
    try:
        ds = xr.open_dataset(f)
        meta["group_label"] = ds.attrs.get("group_label", case)
        meta["family_short"] = ds.attrs.get("family_short", "")
        meta["family"] = ds.attrs.get("family", "")
        ds.close()
    except Exception:
        pass
    return meta


def nice_case_label(meta: dict, n_valid: int) -> str:
    gl = meta.get("group_label", meta["case"])
    fam = meta.get("family_short", "")
    if fam:
        return f"{gl} ({fam}, n={n_valid})"
    return f"{gl} (n={n_valid})"


def load_ref_anom(year_int: int, season: str) -> xr.DataArray:
    if not REF_ANOM_FILE.exists():
        raise FileNotFoundError(f"Missing ref anomaly file: {REF_ANOM_FILE}")
    ds = xr.open_dataset(REF_ANOM_FILE)
    ref = ds["Z500_longrun_anom"]  # (year,season,lat,lon)
    if year_int not in ref["year"].values:
        ds.close()
        raise KeyError(f"Reference missing year={year_int}. Available={ref['year'].values}")
    ref_y = ref.sel(year=year_int)           # (season,lat,lon)
    ref_s = safe_sel_season(ref_y, season)   # (lat,lon)
    ref_s = ref_s.load()
    ds.close()
    return ref_s


def load_hind_ens_anom(case: str, season: str) -> tuple[xr.DataArray, int]:
    f = HIND_ANOM_DIR / f"Z500_anom_fields_{case}.nc"
    if not f.exists():
        raise FileNotFoundError(f"Missing hindcast anomaly file: {f}")

    ds = xr.open_dataset(f)
    z = ds["Z500_hindcast_anom"]  # (member,season,lat,lon)

    if "season_valid" in ds:
        sv = ds["season_valid"]
    else:
        sv = xr.DataArray(
            np.ones((z.sizes["member"], z.sizes["season"]), dtype=np.int8),
            dims=("member", "season"),
            coords={"member": z["member"], "season": z["season"]},
        )

    z_s  = safe_sel_season(z, season)     # (member,lat,lon)
    sv_s = safe_sel_season(sv, season)    # (member,)

    z_s, sv_s = xr.align(z_s, sv_s, join="inner")
    valid = (sv_s == 1)
    n_valid = int(valid.sum().values)

    if n_valid == 0:
        nan2d = xr.full_like(z_s.isel(member=0, drop=True), np.nan)
        ds.close()
        return nan2d.load(), 0

    mem = sv_s["member"].where(valid, drop=True)
    ens = z_s.sel(member=mem).mean("member").load()
    ds.close()
    return ens, n_valid


def compute_common_vlim(fields_2d: list[xr.DataArray], q: float = 0.98) -> float:
    arrs = []
    for f in fields_2d:
        v = f.values
        v = v[np.isfinite(v)]
        if v.size:
            arrs.append(np.abs(v))
    if not arrs:
        return 1.0
    allabs = np.concatenate(arrs)
    vmax = float(np.quantile(allabs, q))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = float(np.nanmax(allabs)) if np.isfinite(allabs).any() else 1.0
    return vmax


def set_circular_boundary(ax):
    theta = np.linspace(0, 2*np.pi, 400)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)


def plot_polar_map(ax, field2d: xr.DataArray, title: str, vlim: float):
    ax.set_title(title, fontsize=FS_TITLE, pad=3)
    ax.set_extent(POLAR_EXTENT, crs=DATA_CRS)
    ax.coastlines(linewidth=COASTLINE_LW)
    ax.add_feature(cfeature.BORDERS, linewidth=BORDER_LW, alpha=0.7)
    set_circular_boundary(ax)

    im = ax.pcolormesh(
        field2d["lon"], field2d["lat"], field2d,
        transform=DATA_CRS,
        cmap=CMAP,
        vmin=-vlim, vmax=vlim,
        shading="auto",
    )
    return im


def plot_one_figure(year_str: str, season: str, df_acc: pd.DataFrame) -> bool:
    season = normalize_season(season)
    year_int = int(year_str)

    # ---------- REF ----------
    try:
        ref2d = load_ref_anom(year_int, season)
    except Exception as e:
        print(f"[SKIP] {year_str} {season}: cannot load reference anomaly: {e}")
        return False

    # ---------- EXPS ----------
    cases_all = list_cases_for_year(year_str)
    exps = []
    for case in cases_all:
        try:
            ens2d, n_valid = load_hind_ens_anom(case, season)
        except Exception as e:
            print(f"  [WARN] {case} {season}: {e}")
            continue
        if n_valid <= 0:
            continue
        meta = load_case_meta(case)
        exps.append({"case": case, "meta": meta, "ens": ens2d, "n_valid": n_valid})

    if len(exps) < MIN_EXPS_PER_FIG:
        print(f"[SKIP] {year_str} {season}: only {len(exps)} valid experiments (<{MIN_EXPS_PER_FIG})")
        return False

    # ACC lookup
    acc_rows = df_acc[(df_acc["year_str"] == year_str) & (df_acc["season"] == season)].copy()
    acc_map = {r["case"]: r["ACC_ensmean"] for _, r in acc_rows.iterrows()}
    for e in exps:
        e["acc"] = float(acc_map.get(e["case"], np.nan))

    # sort display
    exps = sorted(exps, key=lambda e: (e["meta"].get("family_short", ""), e["case"]))

    # vlim shared
    vlim = compute_common_vlim([ref2d] + [e["ens"] for e in exps], q=VLIM_Q)

    # ---------- LAYOUT ----------
    n_maps = 1 + len(exps)
    ncols = MAX_MAPS_PER_ROW
    nrows_maps = int(np.ceil(n_maps / ncols))

    # Bottom bar row height grows with bar count (spacing fix) but figsize unchanged
    bar_row_h = max(BAR_ROW_BASE, BAR_ROW_BASE + BAR_ROW_PER_BAR * max(0, len(exps) - 6))

    height_ratios = [1.0]*nrows_maps + [bar_row_h]

    fig = plt.figure(figsize=FIGSIZE)
    gs = fig.add_gridspec(
        nrows=nrows_maps + 1,
        ncols=ncols,
        height_ratios=height_ratios,
        hspace=0.18,
        wspace=0.06,
    )

    # ---------- MAPS ----------
    im_last = None
    idx = 0
    for k in range(n_maps):
        r = idx // ncols
        c = idx % ncols
        ax = fig.add_subplot(gs[r, c], projection=MAP_PROJ)

        if k == 0:
            ttl = f"REF (BWCN) anom — {year_str} {season}"
            im_last = plot_polar_map(ax, ref2d, ttl, vlim)
        else:
            e = exps[k-1]
            ttl = nice_case_label(e["meta"], e["n_valid"])
            im_last = plot_polar_map(ax, e["ens"], ttl, vlim)

        idx += 1

    # remove unused map axes
    total_slots = nrows_maps * ncols
    for j in range(n_maps, total_slots):
        r = j // ncols
        c = j % ncols
        ax_empty = fig.add_subplot(gs[r, c])
        ax_empty.axis("off")

    # ---------- COLORBAR: RIGHT SIDE, VERTICAL ----------
    # Place a dedicated axes on the right side spanning the map block.
    # This keeps bar panel at bottom untouched.
    # [left, bottom, width, height] in figure fraction
    cax = fig.add_axes([0.92, 0.28, 0.015, 0.55])
    cb = fig.colorbar(im_last, cax=cax, orientation="vertical")
    cb.ax.tick_params(labelsize=FS_TICK)
    cb.set_label("Z500 anomaly (m)", fontsize=FS_LABEL)

    # ---------- ACC BAR (BOTTOM, position unchanged) ----------
    axb = fig.add_subplot(gs[-1, :])

    labels = [e["meta"].get("group_label", e["case"]) for e in exps]
    accs   = np.array([e["acc"] for e in exps], dtype=float)

    # Increase spacing between bars by scaling y positions (bar thickness unchanged)
    y = np.arange(len(labels), dtype=float) * BAR_Y_SPACING

    axb.barh(y, accs, height=BAR_HEIGHT)  # thickness unchanged
    axb.set_yticks(y)
    axb.set_yticklabels(labels, fontsize=FS_TICK)
    axb.invert_yaxis()

    axb.axvline(0.0, linewidth=0.8, alpha=0.8)
    axb.set_xlim(-1.0, 1.0)
    axb.set_xlabel("ACC (ensemble mean)", fontsize=FS_LABEL)
    axb.set_title(f"ACC vs REF ({year_str} {season})", fontsize=FS_TITLE, pad=4)
    axb.tick_params(axis="x", labelsize=FS_TICK)

    # Expand y-limits to ensure full visibility with increased spacing
    if len(y) > 0:
        axb.set_ylim(y.max() + BAR_Y_SPACING*0.8, -BAR_Y_SPACING*0.8)

    # numeric labels at bar end
    for i, v in enumerate(accs):
        if np.isfinite(v):
            x = v + (0.03 if v >= 0 else -0.03)
            ha = "left" if v >= 0 else "right"
            axb.text(x, y[i], f"{v:.3f}", va="center", ha=ha, fontsize=FS_TICK)

    fig.suptitle(f"Z500 anomaly patterns & ACC — {year_str} {season}", fontsize=11, y=0.985)

    out_png = OUT_FIG_DIR / f"lee_style_{year_str}_{season}_reversexy.png"
    out_pdf = OUT_FIG_DIR / f"lee_style_{year_str}_{season}_reversexy.pdf"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    fig.savefig(out_pdf, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"[SAVE] {out_png}")
    return True


def main():
    print("=" * 100)
    print("[INFO] Lee-style plotting: polar projection unchanged + RIGHT vertical colorbar + spaced barh")
    print(f"  HIND_ANOM_DIR: {HIND_ANOM_DIR}")
    print(f"  REF_ANOM_FILE: {REF_ANOM_FILE}")
    print(f"  ACC_CSV      : {ACC_CSV}")
    print(f"  OUT_FIG_DIR  : {OUT_FIG_DIR}")
    print("=" * 100)

    df_acc = load_acc_table()

    print("\n[STEP] Example: 0008 MAM")
    plot_one_figure("0008", "MAM", df_acc)

    print("\n[STEP] Batch all")
    n_done, n_skip = 0, 0
    for y in TARGET_YEARS:
        for s in SEASONS:
            ok = plot_one_figure(y, s, df_acc)
            n_done += int(ok)
            n_skip += int(not ok)

    print("\n" + "=" * 100)
    print(f"[DONE] created: {n_done}, skipped: {n_skip}")
    print(f"[OUT ] {OUT_FIG_DIR}")
    print("=" * 100)


if __name__ == "__main__":
    main()
